# planilha_nereu_aux.ipynb — células novas/de substituição

Patch para `planilha_nereu.ipynb`. **Não roda sozinho**: depende do prelúdio e das células
anteriores (`df_citacoes`, `df_ccd_infos`, `df_notif`) já executadas na mesma sessão/kernel.

Cada célula tem, no topo, um comentário indicando qual célula do `planilha_nereu.ipynb` ela
substitui (ou se é NOVA). Copie-as para o notebook principal — ou rode-as nesta sessão após
as células correspondentes.

**Mudanças:** aba Processos passa a ter grão por par (execução × origem) e ganha
`setor_atual`, `numero_acordao`/`data_acordao` (acórdão mais recente do processo de origem)
e `evento_citacao_5d`; aba Débitos perde `notificado_no_processo`; nova aba
"Notificações desconto em folha" (com a data da notificação da CCD). Todas as datas saem
formatadas como DD/MM/YYYY.


In [ ]:
# PRÉ-REQUISITO — define debitos_nereu SEM cancelados (não existe em planilha_nereu.ipynb).
# Rode antes da célula 12. Equivale ao carregar_debitos corrigido (esd.StatusCancelamento IS NULL).
from ccd.db import run_query_df

SQL_DEBITOS_NEREU = """
SELECT
    e.IdDebito AS id_debito,
    (SELECT CONCAT(p.numero_processo, '/', p.ano_processo)
        FROM processo.dbo.Processos p WHERE p.IdProcesso = e.IdProcessoOrigem) AS processo_origem,
    (SELECT CONCAT(p.numero_processo, '/', p.ano_processo)
        FROM processo.dbo.Processos p WHERE p.IdProcesso = e.IdProcessoExecucao) AS processo_execucao,
    etd.Descricao AS tipo_debito,
    esd.DescricaoStatusDivida AS situacao_divida,
    e.valorOriginalDebito AS valor_original,
    processo.dbo.fn_Exe_RetornaValorAtualizado(e.IdDebito) AS valor_atualizado,
    e.dataTransito AS data_transito
FROM processo.dbo.Exe_Debito e
INNER JOIN processo.dbo.Exe_DebitoPessoa edp ON edp.IDDebito = e.IdDebito
INNER JOIN processo.dbo.GenPessoa gp ON gp.IdPessoa = edp.IDPessoa
LEFT JOIN processo.dbo.Exe_TipoDebito etd ON etd.CodigoTipoDebito = e.CodigoTipoDebito
LEFT JOIN processo.dbo.Exe_StatusDivida esd ON esd.CodigoStatusDivida = e.CodigoStatusDivida
WHERE gp.Documento = :cpf
  AND e.IdDebitoAnterior IS NULL
  AND esd.StatusCancelamento IS NULL
"""


def carregar_debitos(cpf: str = CPF_NEREU) -> pd.DataFrame:
    df = run_query_df(SQL_DEBITOS_NEREU, cpf=cpf)
    df['tipo_debito'] = df['tipo_debito'].fillna('(sem tipo)')
    df['situacao_divida'] = df['situacao_divida'].fillna('(sem situação)')
    return df


debitos_nereu = carregar_debitos()
print(f'Débitos do Nereu (sem cancelados): {len(debitos_nereu)} linhas')
debitos_nereu.head()


In [ ]:
# NOVA — inserir após a célula 11. Metadados por processo: setor atual, número do acórdão
# (mais recente, do processo de origem) e o evento (SequencialProcessoEvento) da citação 5d.
# Define também _fmt_data (usada nas abas para formatar datas como DD/MM/YYYY).
# Consultas baratas (sem LLM). Constrói df_setor, df_acordao e df_citacoes_ev.
from ccd.db import run_query_df


def _fmt_data(s):
    """Formata uma coluna de datas como DD/MM/YYYY (vazio quando NaT)."""
    if not pd.api.types.is_datetime64_any_dtype(s):
        s = pd.to_datetime(s, errors='coerce', format='mixed')
    return s.dt.strftime('%d/%m/%Y').fillna('')


_SUB_EXEC_ORIG = """
    SELECT e.IdProcessoExecucao AS idp FROM processo.dbo.Exe_Debito e
    JOIN processo.dbo.Exe_DebitoPessoa edp ON edp.IDDebito = e.IdDebito
    JOIN processo.dbo.GenPessoa gp ON gp.IdPessoa = edp.IDPessoa
    WHERE gp.Documento = :cpf AND e.IdDebitoAnterior IS NULL
    UNION
    SELECT e.IdProcessoOrigem FROM processo.dbo.Exe_Debito e
    JOIN processo.dbo.Exe_DebitoPessoa edp ON edp.IDDebito = e.IdDebito
    JOIN processo.dbo.GenPessoa gp ON gp.IdPessoa = edp.IDPessoa
    WHERE gp.Documento = :cpf AND e.IdDebitoAnterior IS NULL
"""
_SUB_ORIG = """
    SELECT e.IdProcessoOrigem AS idp FROM processo.dbo.Exe_Debito e
    JOIN processo.dbo.Exe_DebitoPessoa edp ON edp.IDDebito = e.IdDebito
    JOIN processo.dbo.GenPessoa gp ON gp.IdPessoa = edp.IDPessoa
    WHERE gp.Documento = :cpf AND e.IdDebitoAnterior IS NULL
"""

# setor atual (processos de execução + origem)
df_setor = run_query_df(f"""
SELECT DISTINCT CONCAT(p.numero_processo, '/', p.ano_processo) AS processo,
       RTRIM(p.setor_atual) AS setor_atual
FROM processo.dbo.Processos p
WHERE p.IdProcesso IN ({_SUB_EXEC_ORIG})
""", cpf=CPF_NEREU)

# número do acórdão (sessão mais recente) no processo de origem
_ac = run_query_df(f"""
SELECT CONCAT(v.NumeroProcesso, '/', v.AnoProcesso) AS processo_origem,
       v.numeroResultado, v.anoResultado, v.DataSessao
FROM processo.dbo.vw_ia_votos_acordaos_decisoes v
WHERE v.numeroResultado IS NOT NULL
  AND v.IdProcesso IN ({_SUB_ORIG})
""", cpf=CPF_NEREU)
_ac['DataSessao'] = pd.to_datetime(_ac['DataSessao'], errors='coerce')
df_acordao = (
    _ac.sort_values('DataSessao', ascending=False)
       .groupby('processo_origem', as_index=False)
       .head(1)
       .copy()
)
df_acordao['numero_acordao'] = (
    df_acordao['numeroResultado'].astype(str).str.strip() + '/'
    + df_acordao['anoResultado'].astype(str).str.strip()
)
df_acordao = df_acordao.rename(columns={'DataSessao': 'data_acordao'})[
    ['processo_origem', 'numero_acordao', 'data_acordao']]

# evento (SequencialProcessoEvento) da citação 5d, via Pro_ProcessoEvento
_ev = run_query_df(f"""
SELECT ppe.IdInformacao, ppe.SequencialProcessoEvento
FROM processo.dbo.Pro_ProcessoEvento ppe
WHERE ppe.IdInformacao IS NOT NULL
  AND ppe.IdProcesso IN ({_SUB_ORIG})
""", cpf=CPF_NEREU)
_ev = _ev.dropna(subset=['IdInformacao']).drop_duplicates('IdInformacao')
_ev['IdInformacao'] = _ev['IdInformacao'].astype('int64')

df_citacoes_ev = df_citacoes.copy()
df_citacoes_ev['idInformacao_citacao'] = df_citacoes_ev['idInformacao_citacao'].astype('int64')
df_citacoes_ev = df_citacoes_ev.merge(
    _ev.rename(columns={'IdInformacao': 'idInformacao_citacao',
                        'SequencialProcessoEvento': 'evento_citacao_5d'}),
    on='idInformacao_citacao', how='left')

print(f'setor: {len(df_setor)} | acórdãos: {len(df_acordao)} | '
      f'citações c/ evento: {df_citacoes_ev["evento_citacao_5d"].notna().sum()}/{len(df_citacoes_ev)}')
df_citacoes_ev.head()


In [ ]:
# SUBSTITUI a célula 12 (Aba 1 — Totais). Inalterada; opera sobre debitos_nereu sem cancelados.
processos_notificados = set(df_notif['processo'])
mask_notif = debitos_nereu['processo_execucao'].isin(processos_notificados)

qtd_total = len(debitos_nereu)
qtd_notif = int(mask_notif.sum())
val_total = float(debitos_nereu['valor_atualizado'].sum())
val_notif = float(debitos_nereu.loc[mask_notif, 'valor_atualizado'].sum())

tab_totais = pd.DataFrame([
    {'metrica': 'Total de débitos imputados a Nereu',
     'quantidade': qtd_total,
     'valor_atualizado_R$': locale.currency(val_total, grouping=True, symbol=True)},
    {'metrica': 'Total de débitos notificados (desconto em folha, Nereu)',
     'quantidade': qtd_notif,
     'valor_atualizado_R$': locale.currency(val_notif, grouping=True, symbol=True)},
])
tab_totais


In [ ]:
# SUBSTITUI a célula 13 (Aba 2 — Processos). Grão: 1 linha por par (execução, origem).
base = (
    debitos_nereu.groupby(['processo_execucao', 'processo_origem'], dropna=False)
    .agg(
        valor_original=('valor_original', 'sum'),
        valor_atualizado=('valor_atualizado', 'sum'),
        data_transito=('data_transito', 'max'),
    )
    .reset_index()
)

# setor atual: do processo de EXECUÇÃO; se só houver origem (sem execução), usa o da ORIGEM
_setor_map = df_setor.drop_duplicates('processo').set_index('processo')['setor_atual']
base['setor_atual'] = (base['processo_execucao'].map(_setor_map)
                       .fillna(base['processo_origem'].map(_setor_map)))
# acórdão mais recente (processo de origem)
base = base.merge(df_acordao, on='processo_origem', how='left')
# citação 5d + evento (processo de origem)
base = base.merge(
    df_citacoes_ev[['processo_origem', 'data_citacao', 'setor_citacao',
                    'evento_citacao_5d', 'qtd_citacoes_validas']],
    on='processo_origem', how='left')
# notificações CCD (processo de execução)
base = base.merge(
    df_notif[['processo', 'qtd_notificacoes', 'eventos_notificacao_ccd']]
        .rename(columns={'processo': 'processo_execucao'}),
    on='processo_execucao', how='left')

base['qtd_notificacoes'] = base['qtd_notificacoes'].fillna(0).astype(int)
base['qtd_citacoes_validas'] = base['qtd_citacoes_validas'].fillna(0).astype(int)
base['evento_citacao_5d'] = base['evento_citacao_5d'].astype('Int64')
base['eventos_notificacao_ccd'] = base['eventos_notificacao_ccd'].apply(
    lambda v: v if isinstance(v, list) else [])

tab_processos = base[[
    'processo_execucao', 'processo_origem', 'setor_atual',
    'numero_acordao', 'data_acordao', 'data_transito',
    'data_citacao', 'evento_citacao_5d', 'setor_citacao',
    'valor_original', 'valor_atualizado',
    'qtd_citacoes_validas', 'qtd_notificacoes', 'eventos_notificacao_ccd',
]].sort_values('valor_atualizado', ascending=False).copy()

for _c in ('data_acordao', 'data_transito', 'data_citacao'):
    tab_processos[_c] = _fmt_data(tab_processos[_c])

print(f'tab_processos: {len(tab_processos)} pares (execução, origem)')
tab_processos.head()


In [ ]:
# SUBSTITUI a célula 14 (Aba 3 — Débitos). Sem a coluna notificado_no_processo.
tab_debitos = debitos_nereu[[
    'id_debito', 'processo_origem', 'processo_execucao',
    'tipo_debito', 'situacao_divida',
    'valor_original', 'valor_atualizado', 'data_transito',
]].copy().sort_values('valor_atualizado', ascending=False)
tab_debitos['data_transito'] = _fmt_data(tab_debitos['data_transito'])
tab_debitos.head()


In [ ]:
# NOVA — Aba 4 (Notificações desconto em folha): 1 linha por evento de notificação.
# Inclui a data em que a CCD fez a informação de notificação (data_ultima_atualizacao).
# Débitos do processo (execução) agregados em lista (não há vínculo direto evento->débito).
_notif_eventos = (
    df_ccd_infos[df_ccd_infos['eh_notificacao']]
    [['processo', 'evento', 'data_ultima_atualizacao']]
    .drop_duplicates()
    .rename(columns={'processo': 'processo_execucao',
                     'data_ultima_atualizacao': 'data_notificacao'})
)

_deb_por_proc = (
    debitos_nereu.groupby('processo_execucao')
    .agg(
        ids_debitos=('id_debito', lambda s: sorted(int(x) for x in s)),
        valor_original_total=('valor_original', 'sum'),
        valor_atualizado_total=('valor_atualizado', 'sum'),
    )
    .reset_index()
)

tab_notificacoes = (
    _notif_eventos.merge(_deb_por_proc, on='processo_execucao', how='left')
    .rename(columns={'processo_execucao': 'processo'})
)
tab_notificacoes['ids_debitos'] = tab_notificacoes['ids_debitos'].apply(
    lambda v: v if isinstance(v, list) else [])
tab_notificacoes['valor_original_total'] = tab_notificacoes['valor_original_total'].fillna(0.0)
tab_notificacoes['valor_atualizado_total'] = tab_notificacoes['valor_atualizado_total'].fillna(0.0)
tab_notificacoes = tab_notificacoes[[
    'processo', 'evento', 'data_notificacao', 'ids_debitos',
    'valor_original_total', 'valor_atualizado_total',
]].sort_values(['processo', 'evento']).reset_index(drop=True)
tab_notificacoes['data_notificacao'] = _fmt_data(tab_notificacoes['data_notificacao'])

# Dados do PROCESSO (débitos e valores) só na 1ª linha (menor evento) de cada processo
# -> somar valor_atualizado_total passa a bater com o total notificado da aba Totais.
_primeira = ~tab_notificacoes.duplicated('processo')
for _c in ('ids_debitos', 'valor_original_total', 'valor_atualizado_total'):
    tab_notificacoes[_c] = tab_notificacoes[_c].where(_primeira)
print(f"soma valor_atualizado_total (1x/processo): "
      f"{tab_notificacoes['valor_atualizado_total'].sum():.2f}")
print(f'Notificações (eventos): {len(tab_notificacoes)} | '
      f'processos: {tab_notificacoes["processo"].nunique()}')
tab_notificacoes.head()


In [ ]:
# SUBSTITUI a célula 15 (escrita do xlsx) — agora com 4 abas. Datas já em DD/MM/YYYY.
out_xlsx = Path('docs/debitos_nereu_planilha_final.xlsx')
out_xlsx.parent.mkdir(parents=True, exist_ok=True)


def _sanitize(df):
    return df.map(_strip_ctrl) if hasattr(df, 'map') else df.applymap(_strip_ctrl)


# colunas-lista -> string (openpyxl não aceita listas)
tab_processos_xlsx = tab_processos.copy()
tab_processos_xlsx['eventos_notificacao_ccd'] = tab_processos_xlsx['eventos_notificacao_ccd'].apply(
    lambda v: ', '.join(str(x) for x in v) if isinstance(v, list) and v else '')

tab_notificacoes_xlsx = tab_notificacoes.copy()
tab_notificacoes_xlsx['ids_debitos'] = tab_notificacoes_xlsx['ids_debitos'].apply(
    lambda v: ', '.join(str(x) for x in v) if isinstance(v, list) and v else '')

with pd.ExcelWriter(out_xlsx, engine='openpyxl') as writer:
    _sanitize(tab_totais).to_excel(writer, sheet_name='Totais', index=False)
    _sanitize(tab_processos_xlsx).to_excel(writer, sheet_name='Processos', index=False)
    _sanitize(tab_debitos).to_excel(writer, sheet_name='Débitos', index=False)
    _sanitize(tab_notificacoes_xlsx).to_excel(
        writer, sheet_name='Notificações desconto em folha', index=False)

print(f'Planilha salva em: {out_xlsx.resolve()}')
